In [2]:
from sequana import DNA
from sequana import FastA
import pandas as pd
import numpy as np
from sklearn.preprocessing import  StandardScaler
import re
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter
from scipy.signal import argrelextrema
from itertools import combinations

from scipy.signal import peak_widths


In [3]:
def count_homopolymers(seq, min_length=5):
    # Ex : trouve AAAAA ou TTTTT, etc.
    pattern = re.compile(rf"(A{{{min_length},}}|T{{{min_length},}}|C{{{min_length},}}|G{{{min_length},}})")
    return len(pattern.findall(seq.upper()))

def load_fasta(fasta_path, window_size=100):
    f = FastA(fasta_path)
    data = []


    for maseq in f:

        print(maseq.name)
        
        features = []

        s = DNA(maseq.sequence.upper())
        seq = maseq.sequence.upper()
        s.window = window_size

       

        #Homopolymere
        X2 = []
        X3 = []

        
        for i in range(0, len(seq)-2, 1):
            
            window = seq[max(0, i - window_size//2):min(i+window_size//2,len(seq))]
            nb = count_homopolymers(window, min_length=2)
            X2.append(nb)

            nb = count_homopolymers(window, min_length=3)
            X3.append(nb)


            
        X2 = X2[5000:-5000]
        X3 = X3[5000:-5000]



        df= pd.DataFrame({
            'X2':X2,
            'X3':X3
        })

        
        data.append(df)

            
    return data

 
dataBHU1220 = load_fasta("../data/Fasta/Donovani/TriTrypDB-68_LdonovaniBHU1220_Genome.fasta",200)
dataBPK282A1 = load_fasta("../data/Fasta/Donovani/TriTrypDB-68_LdonovaniBPK282A1_Genome.fasta",200)
dataCL = load_fasta("../data/Fasta/Donovani/TriTrypDB-68_LdonovaniCL-SL_Genome.fasta",200)
dataHU3 = load_fasta("../data/Fasta/Donovani/TriTrypDB-68_LdonovaniHU3_Genome.fasta",200)


CM002141.1
CM002142.1
CM002143.1
CM002144.1
CM002145.1
CM002146.1
CM002147.1
CM002148.1
CM002149.1
CM002150.1
CM002151.1
CM002152.1
CM002153.1
CM002154.1
CM002155.1
CM002156.1
CM002157.1
CM002158.1
CM002159.1
CM002160.1
CM002161.1
CM002162.1
CM002163.1
CM002164.1
CM002165.1
CM002166.1
CM002167.1
CM002168.1
CM002169.1
CM002170.1
CM002171.1
CM002172.1
CM002173.1
CM002174.1
CM002175.1
CM002176.1
Ld01_v01s1
Ld02_v01s1
Ld03_v01s1
Ld04_v01s1
Ld05_v01s1
Ld06_v01s1
Ld07_v01s1
Ld08_v01s1
Ld09_v01s1
Ld10_v01s1
Ld11_v01s1
Ld12_v01s1
Ld13_v01s1
Ld14_v01s1
Ld15_v01s1
Ld16_v01s1
Ld17_v01s1
Ld18_v01s1
Ld19_v01s1
Ld20_v01s1
Ld21_v01s1
Ld22_v01s1
Ld23_v01s1
Ld24_v01s1
Ld25_v01s1
Ld26_v01s1
Ld27_v01s1
Ld28_v01s1
Ld29_v01s1
Ld30_v01s1
Ld31_v01s1
Ld32_v01s1
Ld33_v01s1
Ld34_v01s1
Ld35_v01s1
Ld36_v01s1
CP029500
CP029501
CP029502
CP029503
CP029504
CP029505
CP029506
CP029507
CP029508
CP029509
CP029510
CP029511
CP029512
CP029513
CP029514
CP029515
CP029516
CP029517
CP029518
CP029519
CP029520
CP029521
CP029522
C

In [50]:
df = pd.read_csv("../data/Centromere_Positions/pos_libre.csv")
pos_libre = {
    int(row.Chromosome): (int(row.Start), int(row.End))
    for row in df.itertuples(index=False)
}

vecteur_Debut = []
vecteur_Fin = []

vecteur_longeur = []

 
#data = dataBHU1220 
#data = dataBPK282A1 
data = dataCL
#data = dataHU3 

g = 0



for i in range(0,36):
    temp_ =  data[i]['X3']*data[i]['X2']

    telo = 40000
    temp_ = temp_[telo:-1000]

    mean = np.mean(temp_)

    peaks, properties = find_peaks(temp_,  prominence=mean*2, distance=20)

    window = 3000
    peak_medians = []

 


        #Moyenne
    peak_means = []
    for peak in peaks:
        start = max(0, peak - window//2)
        end = min(len(temp_), peak + window//2)
        mean_val = np.mean(temp_[start:end])
        peak_medians.append((peak, mean_val))
    
  #  for peak in peaks:
  #      start = max(0, peak - window//2)
  #      end = min(len(temp_), peak + window//2 + 1)
    
   #     neighborhood = temp_[start:end]
   #     if len(neighborhood) > 0:
   #         median_val = np.median(neighborhood)
   #         peak_medians.append((peak, median_val))
    
    # Trouver le pic avec la médiane la plus élevée

    if peak_medians:
        peak_max, max_median = max(peak_medians, key=lambda x: x[1])
        max_index = peak_max


        #Chercher la taille du centromere


        # Appliquer un filtre pour lisser
        temp_X2_3 = uniform_filter1d(temp_, size=1500)
        temp_X2_3 = temp_X2_3[max_index-25000:max_index+25000]
        
        if i == -12:
            print(temp_X2_3)
            temp_ = temp_[:]
            
            # Initialiser la figure
            fig = go.Figure()
            
            # Ajouter la courbe
            fig.add_trace(go.Scatter(
                y=temp_,
                mode='lines',
                name='Erreur normalisée'
            ))
            
            
            
            
            fig.update_layout(
                title=f"X2-X3 - Chromosome {i}",
                xaxis_title="Index",
                yaxis_title="Erreur normalisée",
                legend=dict(orientation="h")
            )
            fig.show()
        
        # Détection des pics
        peaks, properties = find_peaks(temp_X2_3, prominence=5, distance=200)

            
        # Trouver le pic avec la plus grande **prominence**
        if len(peaks) > 0:
            prominences = properties['prominences']
            peak_index = np.argmax(prominences)
            peak_max = peaks[peak_index]
        
            # Calcule la largeur du pic à mi-hauteur
            widths_result = peak_widths(temp_X2_3, peaks, rel_height=0.4)
        
            taille = widths_result[0][peak_index]
            seuil =  widths_result[1][peak_index]

            debut = widths_result[2][peak_index]
            fin = widths_result[3][peak_index]

            debut = int(debut)
            fin = int(fin)
            
            marge = 750
            finish = True
            distance = []
            temp_fin = []
            while finish:
                
                recherche = False
                limiteFin = min(len(temp_X2_3), fin + marge)
                pos = fin

                while pos < limiteFin and recherche == False:
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos += 1
                if recherche == True:
                     temp_fin.append(fin)
                     distance.append(0)
                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos += 1
                        distance[len(distance)-1] += 1

                     fin = pos
                else: 
                    finish = False

            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                fin = temp_fin[t]
                t = t -1 
            #Avant
            finish = True
            distance = 0
            distance = []
            temp_debut = []

            while finish:
                recherche = False
                limiteDebut = max(0, debut - marge)
                pos = debut
                while pos > limiteDebut and recherche == False:
 
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos -= 1
        
                if recherche == True:
                     temp_debut.append(debut)
                     distance.append(0)

                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos -= 1
                        distance[len(distance)-1] += 1
                     debut = pos
                else:
                    finish = False


            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                debut = temp_debut[t]
                t = t -1 
                




            
            debut = debut+max_index-25000+5000+telo
            fin = fin+max_index-25000+5000+telo
            debut = int(debut)
            fin = int(fin)
            max_index = max_index+5000+telo
            taille = fin - debut

            # Affichage



      
            pos_start, pos_end = pos_libre[i+1]

            if abs(pos_start-debut) > 50000:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille} Attention !!! Diff position : {pos_start-debut}')
            else:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille}')



            vecteur_Debut.append(debut)
            vecteur_Fin.append(fin)
            vecteur_longeur.append(taille)


print(len(vecteur_Debut))
result = pd.DataFrame()
result['Chromosome'] = list(range(1, 37))
result['start'] = vecteur_Debut 
result['end'] = vecteur_Fin
result['length'] = vecteur_longeur
#result.to_csv(f"../output/estimation/donovani/Donovani-BHU1220_X2X3.csv", index=False)
#result.to_csv(f"../output/estimation/donovani/Donovani-BPK282A1_X2X3.csv", index=False)

#result.to_csv(f"../output/estimation/donovani/Donovani-CL_X2X3.csv", index=False)
#result.to_csv(f"../output/estimation/donovani/Donovani-SHU3_X2X3.csv", index=False)




1 : Commence 266899    Fini 268728  Taille : 1829
2 : Commence 270121    Fini 274041  Taille : 3920
3 : Commence 246870    Fini 252398  Taille : 5528
4 : Commence 125325    Fini 131938  Taille : 6613
5 : Commence 364043    Fini 370645  Taille : 6602
6 : Commence 122004    Fini 127254  Taille : 5250
7 : Commence 208898    Fini 214051  Taille : 5153
8 : Commence 449633    Fini 453888  Taille : 4255
9 : Commence 261586    Fini 266513  Taille : 4927
10 : Commence 341272    Fini 346549  Taille : 5277
11 : Commence 158303    Fini 161132  Taille : 2829
12 : Commence 285202    Fini 288663  Taille : 3461
13 : Commence 127564    Fini 130010  Taille : 2446
14 : Commence 166973    Fini 172643  Taille : 5670
15 : Commence 326716    Fini 332212  Taille : 5496
16 : Commence 332526    Fini 336056  Taille : 3530
17 : Commence 342989    Fini 346235  Taille : 3246
18 : Commence 440136    Fini 445909  Taille : 5773
19 : Commence 640192    Fini 646620  Taille : 6428
20 : Commence 520563    Fini 523945  Tai